# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 4 - CTR / Engagement Opportunity Scoring is an **Ranking / Scoring** type of problem as the editor's question is "which page should I open first?", that's an ordering problem.

**Why ranking works:**
- The team can review maybe 20–50 pages. They need the top of the list to be right — Precision@K directly measures that.
- The "target" is a continuous gap (observed CTR minus what's typical for the position tier), so it's naturally a score. No artificial threshold needed.
- The output is a ranked queue with reason codes — exactly what an editor works through.

I've considered the other two suggested and this is why they don't fit

**Why not classification?**
- To classify, I'd need a binary label like "is a CTR opportunity: yes/no." But where does that label come from? If I define it myself, then the model just learns my rule back to me — that's a defined label, not an observed outcome. The skill warns: *a label that comes from someone's rule means your model learns the rule, not the world.*
- A classifier that says "yes" to everything gets 85% accuracy and discovers nothing. The base rate makes it worse. From W01, 85.2% of visible pages already have CTR below 0.5%.

- Even if I used predicted probabilities to rank — I'm just doing ranking with extra steps and a label definition I can't justify.

**Why not clustering?**
- Clusters give groups, not order. If a page lands in the "high-impression, low-CTR" cluster — great, but which page in that cluster should the editor open first? Clustering answers "what kinds of pages exist?" (that's Lane 3). Lane 4 asks "which one first?"
- Silhouette score measures geometric separation, not whether the editor's time was well-spent.






## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target: position-adjusted CTR gap

The target is a **CTR gap score** — how far a page's observed CTR falls below the median CTR for its position tier is represented as

```ctr_gap = tier_median_ctr - page_observed_ctr```

A positive gap means the page is underperforming relative to other pages with similar positions. A bigger gap means a higher priority for review

It's built entirely from **observed** measurements:
- `ctr` - observed clicks / impressions
- `position_tier` — derived from `avg_position`, which is an observed GSC measurement
- The tier median is a summary of observed data, not a rule someone wrote

this is only a **proxy**. The *ideal* target would be: "did CTR increase after an editor reviewed and improved this page?" That's a future observed outcome. But the starter dataset is a single 90-day snapshot with no before/after. So the CTR gap is a **proxy** — it identifies pages that *look like* they're underperforming relative to peers. It doesn't prove that fixing them will help.

**Optional secondary signal: engagement gap**

Pages where the CTR gap is large AND engagement is also weak (engagement_rate < 30%) are doubly flagged. The engagement measurement is also observed (from GA4). Combining both signals into the score makes the ranking stronger without introducing any defined labels.

In [6]:
import pandas as pd
import numpy as np
import os

# ── Load data ─────────────────────────────────────────────────────────────────
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"
GITHUB_URL = "https://raw.githubusercontent.com/JamesIvanMatienzo/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(LOCAL_PATH if os.path.exists(LOCAL_PATH) else GITHUB_URL)

# ── Filter to the analysis slice ──────────────────────────────────────────────
# Only pages with enough impressions and a valid position
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)].copy()

# ── Compute the CTR gap target ────────────────────────────────────────────────
# Tier median CTR = what's "typical" for pages at this position
tier_medians = visible.groupby("position_tier")["ctr"].median().rename("tier_median_ctr")
visible = visible.merge(tier_medians, on="position_tier", how="left")

# CTR gap = how far below the tier median this page sits
visible["ctr_gap"] = visible["tier_median_ctr"] - visible["ctr"]

print("── CTR gap distribution (positive = underperforming vs tier) ──")
print(visible["ctr_gap"].describe().round(4))
print(f"\nPages with positive CTR gap (below tier median): "
      f"{(visible['ctr_gap'] > 0).sum():,} / {len(visible):,} "
      f"({100*(visible['ctr_gap'] > 0).mean():.1f}%)")

print("\n── This target is OBSERVED, not defined ──")
print("  ctr        = clicks_90d / impressions_90d (GSC measurement)")
print("  tier_median = median of observed CTR within position_tier")
print("  ctr_gap    = tier_median - page_ctr (arithmetic on observations)")

── CTR gap distribution (positive = underperforming vs tier) ──
count    16726.0000
mean        -0.0865
std          0.3058
min         -5.2600
25%         -0.1600
50%          0.0000
75%          0.0900
max          0.2400
Name: ctr_gap, dtype: float64

Pages with positive CTR gap (below tier median): 7,950 / 16,726 (47.5%)

── This target is OBSERVED, not defined ──
  ctr        = clicks_90d / impressions_90d (GSC measurement)
  tier_median = median of observed CTR within position_tier
  ctr_gap    = tier_median - page_ctr (arithmetic on observations)


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Precision@20** : out of the top 20 pages the score recommends, how many are genuinely underperforming for their position?

Why this metric and not others:

- **Matches the real constraint.** An editorial team realistically reviews 20–50 pages per cycle. Precision@20 measures exactly that: "were the top 20 recommendations worth opening?"
- **Named before training.** I'm committing to this metric now, before building anything. The skill says: *"Good" defined after the fact always looks good.*
- **Computable on a baseline today.** I can compute Precision@20 for a simple rule (e.g. "sort by raw CTR ascending") right now, giving me something honest to beat.

**Secondary metric: Precision@50** — same idea, larger review batch. Lets me check if the ranking holds deeper in the list.

**What "good" looks like:**
- The starter pipeline's baseline rule gets Precision@50 = 0.240 for the decline-detection task.
- The random forest gets 0.740.
- For my CTR-gap task, I'd consider **Precision@20 ≥ 0.70** a meaningful result — 14 of the top 20 are genuine review candidates. Below 0.50, the ranking isn't earning its keep over a flat sort.

**Validation design:** Client-holdout split — hold out entire clients, score their pages, measure Precision@K on the held-out set. This prevents the model from memorising client-level patterns.

In [7]:
# ── Compute a naive baseline Precision@20 ─────────────────────────────────────
# Baseline: just sort by raw CTR ascending (lowest CTR first)
# "Ground truth": a page is a genuine opportunity if its CTR gap > 0
# AND it has enough engagement data to be actionable (sessions >= 10)

scoreable = visible[visible["sessions_90d"] >= 10].copy()

# Define "genuine opportunity" = below tier median AND has enough volume to matter
scoreable["is_opportunity"] = (
    (scoreable["ctr_gap"] > 0) &
    (scoreable["impressions_90d"] >= 1000)
).astype(int)

print(f"Scoreable pages (>=10 sessions): {len(scoreable):,}")
print(f"Genuine opportunities:           {scoreable['is_opportunity'].sum():,} "
      f"({100*scoreable['is_opportunity'].mean():.1f}%)")

# Baseline 1: sort by raw CTR ascending
baseline_raw = scoreable.sort_values("ctr", ascending=True)
p_at_20_raw = baseline_raw.head(20)["is_opportunity"].mean()
p_at_50_raw = baseline_raw.head(50)["is_opportunity"].mean()

# Baseline 2: sort by CTR gap descending (position-adjusted)
baseline_gap = scoreable.sort_values("ctr_gap", ascending=False)
p_at_20_gap = baseline_gap.head(20)["is_opportunity"].mean()
p_at_50_gap = baseline_gap.head(50)["is_opportunity"].mean()

print("\n── Baseline Precision@K ────────────────────────────────────")
results = pd.DataFrame({
    "Method": ["Raw CTR sort (no adjustment)", "CTR gap sort (position-adjusted)"],
    "Precision@20": [f"{p_at_20_raw:.3f}", f"{p_at_20_gap:.3f}"],
    "Precision@50": [f"{p_at_50_raw:.3f}", f"{p_at_50_gap:.3f}"]
})
print(results.to_string(index=False))
print("\nThis gives us an honest baseline to beat with a learned model.")

Scoreable pages (>=10 sessions): 11,696
Genuine opportunities:           4,204 (35.9%)

── Baseline Precision@K ────────────────────────────────────
                          Method Precision@20 Precision@50
    Raw CTR sort (no adjustment)        0.450        0.600
CTR gap sort (position-adjusted)        0.650        0.520

This gives us an honest baseline to beat with a learned model.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### One row = one page (content item)

The unit of analysis is a **single page** — one pseudonymised content item identified by `content_id`. Each row carries:

- **Who it belongs to:** `client_id` (for holdout splits, never as a feature)
- **What it is:** `content_type`, `main_intent`, `word_count`, `content_age_days`
- **How visible it is:** `impressions_90d`, `avg_position`, `position_tier`
- **How well it converts visibility to clicks:** `ctr`, `clicks_90d`
- **How well it engages visitors:** `engagement_rate`, `scroll_rate`, `sessions_90d`
- **The target:** `ctr_gap` (tier_median_ctr − page_ctr)

The slice: only pages with `impressions_90d >= 500` and `avg_position > 0` (valid position data). This filters out pages with too little visibility to matter and pages with no position measurement.

In [8]:
show_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "position_tier", "engagement_rate", "scroll_rate",
    "sessions_90d", "word_count", "content_age_days",
    "tier_median_ctr", "ctr_gap"
]

print(f"Analysis slice: {len(visible):,} rows x {len(show_cols)} columns")
print(f"One row = one page (content item)")
print(f"Clients in slice: {visible['client_id'].nunique()}")
print()

# Show 5 example rows — sorted by CTR gap so the top opportunities are visible
print("── Top 5 pages by CTR gap (biggest opportunities first) ─────")
display_df = visible.sort_values("ctr_gap", ascending=False)[show_cols].head(5)
print(display_df.to_string(index=False))

Analysis slice: 16,726 rows x 16 columns
One row = one page (content item)
Clients in slice: 28

── Top 5 pages by CTR gap (biggest opportunities first) ─────
          content_id         client_id    content_type   main_intent  impressions_90d  clicks_90d  ctr  avg_position position_tier  engagement_rate  scroll_rate  sessions_90d  word_count  content_age_days  tier_median_ctr  ctr_gap
content_9983d31c53cb client_4e07408562 keyword article transactional             7737           0  0.0           5.5        page_1              0.0        14.29             7      2396.0               326             0.24     0.24
content_bb485e06e530 client_19581e27de keyword article transactional             2080           0  0.0           6.3        page_1              0.0         0.00            11      3423.0               139             0.24     0.24
content_cab6d15a5215 client_f74efabef1 keyword article informational             2092           0  0.0           6.5        page_1              0.0 

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like "flag every page with CTR < 0.5%" catches 14,245 of 16,726 visible pages — that's 85% of the pool. An editor can't review 14,000 pages. Even the position-adjusted CTR gap is only one dimension. The pattern that separates a *genuinely fixable* opportunity from noise involves multiple signals interacting:

1. **Position × CTR interaction.** From W01, top_3 pages actually have *lower* median CTR (0.200%) than page_1 pages (0.240%). A rule that assumes "better position = higher expected CTR" would get this backwards. The relationship between position and CTR isn't monotonic — it depends on the query mix, SERP features, and competition at that position.

2. **Content type × intent × CTR.** A keyword article with informational intent at position 5 has a different expected CTR than a comparison article with commercial intent at the same position. A single threshold can't account for both.

3. **Volume × gap × engagement.** A page with 50,000 impressions and a 0.1% CTR gap is more valuable to fix than a page with 600 impressions and a 0.3% CTR gap. But "just multiply gap × volume" doesn't work either — a huge page with a tiny gap and great engagement probably doesn't need attention. The interaction of volume, gap size, and engagement quality is genuinely multi-dimensional.

4. **Freshness and age context.** A brand-new page (90 days old) with low CTR might just need time to settle. A 365-day-old page with low CTR has a more stable signal. Age interacts with everything else.

**The test:** The code cell below shows that a flat CTR threshold rule and a simple CTR-gap sort disagree substantially on their top 20. A learned model can weight these interacting signals to produce a tighter ranking.

In [9]:
# ── Show that a flat rule and the CTR gap disagree on priorities ───────────────

# Rule 1: lowest raw CTR first
top20_raw = set(scoreable.sort_values("ctr").head(20)["content_id"])

# Rule 2: largest CTR gap first (position-adjusted)
top20_gap = set(scoreable.sort_values("ctr_gap", ascending=False).head(20)["content_id"])

overlap = len(top20_raw & top20_gap)
print(f"Top 20 by raw CTR:       20 pages")
print(f"Top 20 by CTR gap:       20 pages")
print(f"Overlap:                 {overlap} pages ({100*overlap/20:.0f}%)")
print(f"Disagree on:             {20 - overlap} pages ({100*(20-overlap)/20:.0f}%)")
print()
print("These two simple rules already disagree on most of their top 20.")
print("Adding content type, intent, engagement, age, and volume makes")
print("the interaction space too large for any single if-statement.")
print("A learned scoring model can weigh all these signals together.")

# ── Show the disagreement in detail ───────────────────────────────────────────
only_raw = top20_raw - top20_gap
only_gap = top20_gap - top20_raw

if only_raw:
    print("\n── Pages in raw-CTR top 20 but NOT in gap top 20 ──────────")
    raw_only_df = scoreable[scoreable["content_id"].isin(only_raw)][
        ["content_id", "ctr", "avg_position", "position_tier", "impressions_90d", "ctr_gap"]
    ].sort_values("ctr")
    print(raw_only_df.head(5).to_string(index=False))
    print("  (These pages have very low CTR but may sit at deep positions")
    print("   where low CTR is normal — the flat rule flags them, the gap doesn't.)")

Top 20 by raw CTR:       20 pages
Top 20 by CTR gap:       20 pages
Overlap:                 1 pages (5%)
Disagree on:             19 pages (95%)

These two simple rules already disagree on most of their top 20.
Adding content type, intent, engagement, age, and volume makes
the interaction space too large for any single if-statement.
A learned scoring model can weigh all these signals together.

── Pages in raw-CTR top 20 but NOT in gap top 20 ──────────
          content_id  ctr  avg_position position_tier  impressions_90d  ctr_gap
content_cfbb81c693df  0.0          13.7      striking             4540     0.17
content_eecb0c9bd176  0.0          31.1      page_3_5              745     0.09
content_79fa5e5f69c8  0.0          12.9      striking             2218     0.17
content_5f89b21c2b57  0.0           7.9        page_1              849     0.24
content_1265114912b4  0.0           9.8        page_1              860     0.24
  (These pages have very low CTR but may sit at deep position

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.